<style>
    pre {
        white-space: pre-wrap;
        word-wrap: break-word;
    }
</style>

<div style="display:flex; justify-content:space-around; align-items:center; background-color:#cccccc; padding:5px; border:2px solid #333333;">
    <a href="https://www.um.es/web/estudios/grados/ciencia-ingenieria-datos/" target="_blank">
    <img src="https://www.um.es/documents/1073494/42130150/LogosimboloUMU-positivo.png" alt="UMU" style="height:200px; width:auto;">
    <a href="https://estudios.upct.es/grado/5251/inicio" target="_blank">
    <img src="https://www.upct.es/contenido/universidad/galeria/identidad-2021/logos/logos-upct/marca-upct/marca-principal/horizontal/azul.png" alt="UPCT" style="height:145px; width:auto;">
</div>

# Asignatura: **Deep Learning**

## Titulación: **Grado en Ciencia e Ingeniería de Datos**

## Práctica 3: Transformers
### **Sesión 3/3: Transformer**

**Autores**: Antonio Martínez Sánchez, Juan Morales Sánchez, José Luís Sancho Gómez y Juan Antonio Botía Blaya

<div style="page-break-before: always;"></div>

### Contenidos
- [Requisitos](#requisitos)
- [Transformer](#transformer)
- [Embedding posicional](#embedp)
- [Ejercicios](#ejercicios)

### Requisitos
<a class='anchor' id='requisitos'></a>

Se trabajará con notebooks de [Jupyter](https://jupyter.org/install) con código Python empleando como intérprete la última versión de [Miniconda](https://docs.anaconda.com/miniconda/). Se requiere la preinstalación (se recomienda utilizar [pip](https://pypi.org/project/pip/)) de los siguientes paquetes de Python:

- [Numpy](https://pypi.org/project/numpy/) (computación numérica)
- [Tensorflow](https://www.tensorflow.org/install/pip?hl=es-419#linux) que incluye a Keras (deep learning)
- [Scikit-learn](https://pypi.org/project/scikit-learn/) (machine learning)
- [Matplotlib](https://pypi.org/project/matplotlib/) (visualización de datos)
- [Pandas](https://pypi.org/project/seaborn/) (manipulación de datos tabulados)

### Transformer
<a class='anchor' id='transformer'></a>

Las redes RNN son difíciles de entrenar, aun así, los LSTM han sido la arquitectura más utilizada para la traducción de texto hasta la llegada de los transformers allá por el 2017. Los transformers implementan de forma práctica el concepto de multi-atención. Durante el *embedding* hemos generado un espacio que captura la relación entre los tokens vecinos analizando una serie de secuencias. Sin embargo, este proceso no permite representar relaciones más complejas. Volviendo al caso del lenguaje natural, el significado de una palabra es específico de su contexto. Por ejemplo, la palabra "estación" tiene un significado diferente en las frases "el tren salió de la estación" o "el verano es la estación más calurosa del año". Otro ejemplo claro son los pronombres "él", "esto", "aquello", ..., que pueden variar su significado en una misma frase. Por esto un modelo debe aprender los diferentes contextos y saber cuándo aplicarlos a cada token. Los Transformes surgieron para conseguir este propósito, para ello utilizan el mecanismo de atencián asignando a cada instancia de un token una semántica en función de su contexto. Además, la multi-atención permite factorizar en espacio de características en diferentes subespacios, lo cual había mostrado un buen comportamiento con las redes CNN. 

En esta sesión vamos a implementar un encoder basado en un Tranformer, para ello construiremos una clase que herede de [Layer](https://keras.io/api/layers/base_layer/) de Keras. Tal y como viene explicado en la documentación de Keras, es necesario implementar el constructor dónde se inicialicen las variables de clase y el método *call()* que será ejecutado durante el entrenamiento y la inferencia. Además, también tendremos que implementar el método *get_config()* para almacenar la variables de clase durante la serialización, necesaria para almacenar el modelo en disco. El Transformer encoder a implementar se basará en siguiente esquema:

<img src="./figs/trans_encoder.png" width="400">

Keras tiene implementadas clases para las capas [MultiHeadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/), [LayerNormalization](https://keras.io/api/layers/normalization_layers/layer_normalization/) y [Dense](https://keras.io/api/layers/core_layers/dense/). La capa MultiHeadAttention requiere de tres entradas: query, key and values. Es importante resaltar que en este caso, puesto que se trata de un encoder, las tres entradas de la capa MultiHeadAttention se corresponden con la única entrada del encoder. También es importante remarcar que este capa recibirá la entrada de una capa de embedding que generará un máscara, esta máscara tiene que ser recogida por el método *call()* del transformer encoder implementado y transformada para que dimensionalmente encaje con las dimensiones requeridas por la capa MultiHeadAttention.  

### Embedding posicional
<a class='anchor' id='codep'></a>

Un transformer como el de la sección anterior por sí solo es insensible a la posición de los tokens en la secuencia, como el MLP. No obstante, cuando introdujimos el LSTM, ya comentamos que una de sus ventajas era que las redes RNN son sensibles al orden de la secuencia. Así que se necesita que la entrada contenga dos informaciones, la semántica de cada token en función de su relación con los tokens vecinos (embedding) y la posición del token en la secuencia. Para codificar la información sobre la posición podría bastar con crear un índice, pero en la práctica no es una buena idea porque a las redes neuronales no les gusta los números grandes o las distribuciones discretas. En el árticulo original dónde se presentaron los transformers [Attention Is All You Need](https://arxiv.org/abs/1706.03762) emplea una función trigonométrica para codificar la posición en el rango -1 y 1. Aquí emplearemos una aproximación más simple pero efectiva, crearemos un array con las posiciones de cada token y las embeberemos en un espacio vectorial, después los vectores embebidos de posición se sumarán a los vectores embebedidos de los tokens. Esta técnica, conocida como *embedding posicional*, se resume en el siguiente esquema:

<img src="./figs/pos_embedding.png" width="400">

La clase que implemente el esquema de arriba ha de implementar un método llamado *compute_mask* que será invocado por el framework. Como partiremos de una vectorización tipo *Integer* con secuencias de diferente tamaño, la máscara servirá para definir qué posiciones de la secuencia contienen AAs.

### Ejercicios
<a class='anchor' id='embedp'></a>

**E1:** Implementa la clase MyTransformerEncoder según el esquema descrito anteriormente.

**E2:** Construye un clasificador a partir de MyTransformerEncoder. Para ello aplica una capa de [Embedding](https://keras.io/api/layers/core_layers/embedding/) antes de pasar los datos a MyTransfomerEncoder, luego aplica un [GlobalMaxPooling1D](https://keras.io/api/layers/pooling_layers/global_max_pooling1d/) a la salida y una capa final de activación sigmoide.

**E3:** Carga el dataset de los péptidos y entrena el codificador completo del E2 y evalúa su precisión.

**E4:** Implementa la clase MyPositionEmbedding según el esquema descrito anteriormente.

**E5:** Construye un clasificador sustituyendo la capa de Embedding por la clase implementada en E4. Entrena y evalúa el nuevo modelo. ¿Qué nueva información se ha añadidio al clasificador? ¿ha mejorado los resultados? ¿por qué?

**E6:** ¿Por qué parece que la mejora de utilizar la información posicional no es muy significativa? ¿a qué se podría deber?

**E7:** Ajusta los hiperparámetros del modelo en E5 y el LSTM bidireccional del E4 de la Sesión 2 hasta obtener una configuración óptima para cada modelo. Utiliza el clúster [ATLAS](https://scc.atica.um.es/intro.html) para acelerar el entrenamiento y testeo de los modelos. ¿Qué modelo consigue un mejor funcionamiento? ¿qué puedes decir sobre el entrenamiento de ambos modelos? 

El embedding posicional puede no mejorar los resultados por varias razones:

- **Naturaleza de la tarea:** Si la tarea no depende fuertemente del orden de los elementos (por ejemplo, clasificaciones basadas en la presencia de ciertos tokens, más que en su secuencia), la información posicional puede aportar poco valor.

- **Longitud y complejidad de las secuencias:** En secuencias muy cortas o cuando la relación de orden no es crucial, la ventaja de conocer la posición es mínima.

- **Diseño del modelo:** Es posible que otros componentes del modelo (como capas de atención) ya capten relaciones contextuales de manera efectiva, haciendo redundante el aporte del embedding posicional.

- **Implementación o hiperparámetros:** Si el método para incorporar la posición (por ejemplo, la forma de combinar el embedding posicional con el de tokens) no se ajusta bien o si los hiperparámetros no están optimizados, el beneficio puede ser nulo o incluso contraproducente.

En resumen, el embedding posicional mejora el rendimiento cuando el orden de los tokens es esencial para la tarea, pero si la tarea o el diseño del modelo ya capturan esas relaciones o si las secuencias no requieren esa información, su impacto puede ser limitado.

In [1]:
# ===================================================
# Imports + semilla
# ===================================================
import random
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"


def set_seed(seed=42):
    """
    Fija la semilla en NumPy, random de Python y Tensorflow.

    Objetivo:
    - Reducir variabilidad entre ejecuciones.
    - Facilitar comparación justa entre modelos.
    """
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


set_seed(42)

I0000 00:00:1775662859.611555   61654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775662859.649195   61654 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775662860.629401   61654 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ===================================================
# Carga de datos + split + codificación integer
# ===================================================
# Esta cela:
# 1) Carga los CSV de AMO y no-AMP
# 2) Limpia secuencias.
# 3) Crea split estratificado train/test.
# 4) Vectoriza a índices con padding fijo.

non_amp_path = "data/in/non_amp_ampep_cdhit90.csv"
amp_path = "data/in/veltri_dramp_cdhit_90.csv"

# Leemos ambos CSV
df_non = pd.read_csv(non_amp_path, index_col=0)
df_amp = pd.read_csv(amp_path, index_col=0)

# Unimos los dos DataFrames en uno solo.
df = pd.concat([df_non, df_amp], ignore_index=True)

# Limpiamos secuencias:
# - convertimos a string
# - eliminamos espacios en extremos
# - normalizamos a mayúsculas
df["aa_seq"] = df["aa_seq"].astype(str).str.strip().str.upper()

# Aseguramos etiqueta binaria como entero
df["AMP"] = df["AMP"].astype(int)

# X: secuencias de aminoácidos (texto)
# y: etiqueta 0/1 (no AMP / AMP).
X = df["aa_seq"].to_numpy()
y = df["AMP"].to_numpy()

# Split stratificado
# mantiene proporción de clases en train y test.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Longitud máxima de secuencia en train.
# Se usa para fijar el padding de todas las muestras.
max_len = max(len(seq) for seq in X_train_text)

# TextVectorization en modo "int":
# cada carácter AA se transforma en un índice entero.
vectorizer_int = layers.TextVectorization(
    standardize=None,  # no modifica texto (respetar AA tal cual)
    split="character",  # tokenización carácter a carácter
    output_mode="int",  # salida: índices enteros
    output_sequence_length=max_len,  # padding/truncado a longitud fija
)

# Aprendemos vocabulario usando SOLO train para evitar fuga de información.
vectorizer_int.adapt(X_train_text)

# Convertimos train/test a matrices enteras.
X_train_int = vectorizer_int(tf.constant(X_train_text)).numpy()
X_test_int = vectorizer_int(tf.constant(X_test_text)).numpy()

# Guardamos vocabulario y tamaño.
vocab_int = vectorizer_int.get_vocabulary()
vocab_size = len(vocab_int)

print("=== Datos preparados ===")
print("Train shape:", X_train_int.shape)
print("Test shape:", X_test_int.shape)
print("vocab_size:", vocab_size)
print("max_len:", max_len)

W0000 00:00:1775662861.447257   61654 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1775662861.459592   61654 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1775662861.579433   61654 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1775662861.580705   61654 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5732 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


=== Datos preparados ===
Train shape: (3178, 101)
Test shape: (795, 101)
vocab_size: 26
max_len: 101


In [3]:
# ===================================================
# E1: MyTransformerEncoder
# ===================================================
# Arquitectura implementada (según esquema):
# Entrada -> MultiHeadAttention -> Residual + LayerNorm ->
# Proyección densa (2 Dense) -> Residual + LayerNorm -> Salida
#
# Detalles clave:
# - Soporte de máscara (supports_masking=True).
# - Transformación de máscara para MultiHeadAttention.
# - get_config para poder serializar/guardar modelo.


@keras.utils.register_keras_serializable(package="custom")
class MyTransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, dropout=0.1, **kwargs):
        """
        Parámetros:
        - embed_dim: dimensión del embedding de entrada/salida.
        - dense_dim: dimensión oculta de la MLP interna del bloque.
        - num_heads: número de cabezas de atención.
        - dropout: regularización para atención y proyección.
        """
        super().__init__(**kwargs)

        # Comprobación importante:
        # para dividir embedding entre cabezas, embed_dim debe ser divisible.
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim debe ser divisible por num_heads.")

        # Guardamos hiperparámetros para serialización
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.dropout = dropout

        # Indicamos a Keras que esta capa entiende entiende máscaras.
        self.supports_masking = True

        # Capa de auto-atención multi-cabeza.
        # key_dim es la dimensión por cabeza.
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=self.embed_dim // num_heads,
            dropout=dropout,
        )

        # Proyección densa tipo "feed-forward network" del Transformer.
        # Primera Dense expande y aplica no linealidad.
        # Segunda Dense vuelve a embed_dim para poder hacer residual.
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )

        # Dropout para regularizar después de atención y proyección.
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

        # Normalizaciones de capa tras cada suma residual.
        self.layernorm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, mask=None, training=None):
        """
        Flujo forward del bloque encoder.

        inputs shape esperado:  (batch, seq_len, embed_dim)
        mask shape esperado:    (batch, seq_len), con True donde hay token real.
        """
        attention_mask = None

        # Si hay máscara (proveniente de Embedding con mask_zero=True)
        # la convertimos al formato que espera MultiHeadAttention.
        # De (batch, seq_len) a (batch, 1, seq_len), para broadcasting.
        if mask is not None:
            attention_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.bool)

        # Auto-atención: query=key=value=inputs (encoder).
        attn_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            training=training,
        )

        # Regularización tras atención
        attn_output = self.dropout_1(attn_output, training=training)

        # Primera conexión residual + normalización
        out_1 = self.layernorm_1(inputs + attn_output)

        # Proyección densa del bloque Transformer.
        proj_output = self.dense_proj(out_1, training=training)
        proj_output = self.dropout_2(proj_output, training=training)

        # Segunda conexión residual + normalización
        return self.layernorm_2(out_1 + proj_output)

    def compute_mask(self, inputs, mask=None):
        """
        Reenviamos la máscara hacia capas posteriores.
        Esto permite que otras capas compatibles la usen.
        """
        return mask

    def get_config(self):
        """
        Permite serializar la capa (guardar/cargar modelo completo).
        """
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
                "dropout": self.dropout,
            }
        )
        return config

In [4]:
# ===================================================
# E2: Clasificador con Embedding + Transformer
# ===================================================

def make_callbacks():
    """
    Devuelve callbacks estándar para:
    - detener temprano si no mejor val_auc,
    - reducir learning rate cuando la validación se estanca.
    """
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-5,    
        )
    ]

def build_transformer_classifier(
    vocab_size,
    max_len,
    embed_dim=64,
    dense_dim=128,
    num_heads=4,
    dropout=0.2,
    lr=1e-3,
    use_positional=False,
):
    """
    Construye el clasificador:
    Entrada integer -> Embedding -> MyTransformerEncoder ->
    GlobalMaxPooling1D -> Dropout -> Dense(sigmoid)

    Si use_positional=True, usa MyPositionEmbedding en lugar de Embedding. 
    """
    # Entrada de enteros con longitud fija max_len.
    inputs = keras.Input(shape=(max_len,), dtype="int32")

    # E2/E3: embedding estándar (sin posición explícita).
    # E5: embedding posicional aprendido.
    if use_positional:
        x = MyPositionEmbedding(
            sequence_length=max_len,
            input_dim=vocab_size,
            output_dim=embed_dim,
            mask_zero=True,
        )(inputs)
    else:
        x = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            mask_zero=True,
        )(inputs)

    # Bloque Transformer encoder personalizado.
    x = MyTransformerEncoder(
        embed_dim=embed_dim,
        dense_dim=dense_dim,
        num_heads=num_heads,
        dropout=dropout,
    )(x)

    # Pooling global para convertir secuencia -> vector fijo
    x = layers.GlobalMaxPooling1D()(x)

    # Dropout final antes de clasificación binaria
    x = layers.Dropout(dropout)(x)

    # Capa de salida para probabilidad AMP.
    outputs = layers.Dense(1, activation="sigmoid")(x)

    # Construcción y compilación del modelo final.
    model = keras.Model(inputs, outputs, name="transformer_classifier")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model

In [5]:
# ===================================================
# E3: Entrenamiento y evaluación del modelo E2
# ===================================================
# E2: arquitectura con Embedding + Transformer + GlobalMaxPooling + Sigmoid
# E3: entrenamiento con dataset de péptidos y evaluación en test.

# Limpiamos estado de Keras y semilla para ejecución consistente.
keras.backend.clear_session()
set_seed(42)

# Construimos modelo sin codificación posicional explícita.
model_e3 = build_transformer_classifier(
    vocab_size=vocab_size,
    max_len=max_len,
    embed_dim=64,
    dense_dim=128,
    num_heads=4,
    dropout=0.2,
    lr=1e-3,
    use_positional=False, # clave de E2/E3
)

# Entrenamiento.
# validation_split=0.2 usa 20% de train como validación
history_e3 = model_e3.fit(
    X_train_int,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    callbacks=make_callbacks(),
    verbose=1
)

# Evaluación final en test.
metrics_e3 = model_e3.evaluate(X_test_int, y_test, return_dict=True, verbose=0)

print("=== E3: Métricas en test (sin posición) ===")
for k, v in metrics_e3.items():
    print(f"{k}: {v:.4f}")

Epoch 1/40


/home/pyros05/anaconda3/envs/DL/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'global_max_pooling1d' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
I0000 00:00:1775662864.529239   61789 service.cc:153] XLA service 0x759b38006b50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775662864.529256   61789 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0a (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.18.1)
I0000 00:00:1775662864.582858   61789 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775662864.911611   61789 cuda_dnn.cc:461] Loaded cuDNN version 91801
I0000 00:00:1775662865.005069   61

24/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4713 - auc: 0.4824 - loss: nan     

I0000 00:00:1775662880.394493   61789 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_253', 228 bytes spill stores, 228 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'fusion_251', 156 bytes spill stores, 208 bytes spill loads

I0000 00:00:1775662880.423007   61789 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


37/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4727 - auc: 0.4870 - loss: nan

...

I0000 00:00:1775662904.234019   61787 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_55', 188 bytes spill stores, 244 bytes spill loads



40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 0.0010
Epoch 3/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 0.0010
Epoch 4/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 5/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 6/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 2.5000e-04


I0000 00:00:1775662905.744997   61782 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6059__.20
I0000 00:00:1775662906.801906   63800 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1775662910.091336   61782 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_55', 188 bytes spill stores, 244 bytes spill loads

I0000 00:00:1775662910.255651   61772 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6059__.20
I0000 00:00:1775662910.704854   63965 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11', 4 bytes spill stores, 4 bytes spill loads



=== E3: Métricas en test (sin posición) ===
accuracy: 0.4805
auc: 0.5000
loss: nan


I0000 00:00:1775662914.699397   61772 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_55', 188 bytes spill stores, 244 bytes spill loads



In [6]:
# ===================================================
# E4: MyPositionEmbedding
# ===================================================
# Idea:
# - Embedding de tokens (AA) + Embedding de posiciones.
# - Ambos vectores se suman elemento a elemento.
#
# Además:
# - compute_mask propaga máscara de padding (token 0).
# - get_config permite se suman elemento.

@keras.utils.register_keras_serializable(package="custom")
class MyPositionEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, mask_zero=True, **kwargs):
        """
        Parámetros:
        - sequence_length: longitud máxima de secuencia (max_len).
        - input_dim: tamaño de vocabulario.
        - output_dim: dimensión del embedding.
        - mask_zero: si True, el token 0 se interpreta como padding.
        """
        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.mask_zero = mask_zero
        self.supports_masking = True

        # Embedding semántico de tokens.
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim,
            output_dim=output_dim,
            mask_zero=mask_zero,
        )

        # Embedding de posiciones absolutas [0, 1, ..., sequence_length-1]
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=output_dim
        )

    def call(self, inputs):
        """
        inputs: tensor (batch, seq_len) con índices enteros.
        salida: tensor (batch, seq_len, output_dim)
        """
        # Longitud dinámica real de la secuencia (normalmente max_len).
        length = tf.shape(inputs)[-1]

        # Índices de posición 0..length-1.
        positions = tf.range(start=0, limit=length, delta=1)

        # Embedding de tokens: depende del contenido.
        embedded_tokens = self.token_embeddings(inputs)

        # Embedding posicional: depende del índice de posición
        embedded_positions = self.position_embeddings(positions)

        # Suma token + posición (broadcast en dimensión batch)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        """
        Reutiliza la máscara calculada por token_embeddings.
        Así el padding (0) se ignora en capas posteriores compatibles.
        """
        return self.token_embeddings.compute_mask(inputs)

    def get_config(self):
        """
        Configuración serializable de la capa.
        """
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "input_dim": self.input_dim,
                "output_dim": self.output_dim,
                "mask_zero": self.mask_zero
            }
        )
        return config

In [7]:
# ===================================================
# E5: Clasificador sustituyendo Embedding por MyPositionEmbedding
# ===================================================
# Se repite la misma arquitectura, sustituyendo la capa Embedding
# por MyPositionEmbedding para añadir información de orden explícita.

keras.backend.clear_session()
set_seed(42)

model_e5 = build_transformer_classifier(
    vocab_size=vocab_size,
    max_len=max_len,
    embed_dim=64,
    dense_dim=128,
    num_heads=4,
    dropout=0.2,
    lr=1e-3,
    use_positional=True,  # clave de E5
)

history_e5 = model_e5.fit(
    X_train_int,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    callbacks=make_callbacks(),
    verbose=1,
)

metrics_e5 = model_e5.evaluate(X_test_int, y_test, return_dict=True, verbose=0)

print("=== E5: Métricas en test (con posición) ===")
for k, v in metrics_e5.items():
    print(f"{k}: {v:.4f}")

# Comparación directa E5 vs E3.
delta_acc = metrics_e5["accuracy"] - metrics_e3["accuracy"]
delta_auc = metrics_e5["auc"] - metrics_e3["auc"]

print("\n=== Comparación E5 - E3 ===")
print(f"Diferencia en accuracy: {delta_acc:+.4f}")
print(f"Diferencia en auc     : {delta_auc:+.4f}")

Epoch 1/40


I0000 00:00:1775662917.084265   61772 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13846__.56


38/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4780 - auc: 0.4925 - loss: nan   

I0000 00:00:1775662925.635174   61772 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_256', 228 bytes spill stores, 228 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'fusion_254', 156 bytes spill stores, 208 bytes spill loads

I0000 00:00:1775662926.234054   61777 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13846__.56


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.4782 - auc: 0.4928 - loss: nan

...

I0000 00:00:1775662940.393874   61782 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_57', 188 bytes spill stores, 244 bytes spill loads



40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 0.0010
Epoch 3/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 0.0010
Epoch 4/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 5/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 6/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4823 - auc: 0.5000 - loss: nan - val_accuracy: 0.4717 - val_auc: 0.5000 - val_loss: nan - learning_rate: 2.5000e-04


I0000 00:00:1775662941.868181   61787 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_14713__.20
I0000 00:00:1775662944.469325   61787 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_57', 188 bytes spill stores, 244 bytes spill loads

I0000 00:00:1775662944.603826   61787 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_14713__.20


=== E5: Métricas en test (con posición) ===
accuracy: 0.4805
auc: 0.5000
loss: nan

=== Comparación E5 - E3 ===
Diferencia en accuracy: +0.0000
Diferencia en auc     : +0.0000


I0000 00:00:1775662947.253915   61787 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_57', 188 bytes spill stores, 244 bytes spill loads



In [8]:
# ===================================================
# E7: Comparación justa con BiLSTM + ajuste de hiperparámetros
# ===================================================


def build_bilstm_classifier(
    vocab_size,
    max_len,
    embed_dim=32,
    lstm_units=64,
    dropout=0.2,
    lr=1e-3,
):
    """
    Modelo BiLSTM de referencia para comparación con Transformer.
    """
    inputs = keras.Input(shape=(max_len,), dtype="int32")
    x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(inputs)
    x = layers.Bidirectional(layers.LSTM(lstm_units, dropout=dropout))(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs, name="bilstm_classifier")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )
    return model


def train_eval_once(
    model, X_train, y_train, X_test, y_test, epochs=40, batch_size=64
):
    """
    Entrena un modelo una sola vez y devuelve métricas útiles:
    - métricas de test,
    - mejor val_accuracy y mejor val_auc alcanzadas,
    - tiempo total de entrenamiento.
    """
    callbacks = make_callbacks()

    t0 = time.time()

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )

    train_time = time.time() - t0
    test_metrics = model.evaluate(X_test, y_test, return_dict=True, verbose=0)

    result = {
        "accuracy": float(test_metrics["accuracy"]),
        "auc": float(test_metrics["auc"]),
        "best_val_accuracy": float(np.max(history.history["val_accuracy"])),
        "best_val_auc": float(np.max(history.history["val_auc"])),
        "train_time_s": float(train_time),
    }
    return result


def run_grid(builder_fn, grid, repeats=2, base_seed=42):
    """
    Ejecuta una búsqueda de hiperparámetros simple por rejilla.

    Parámetros:
    - builder_fn: función que recibe hiperparámetros y devuelve modelo compilado.
    - grid: lista de diccionarios con configuraciones.
    - repeats: número de repeticiones por configuración (estabilidad).
    - base_seed: semilla base; se incrementa por repetición.

    Devuelve:
    - DataFrame ordenado por auc_mean descendente.
    """
    rows = []

    for cfg in grid:
        all_runs = []

        for r in range(repeats):
            # Reiniciamos estado y semilla en cada repetición
            # para comparar configuraciones de forma más limpia.
            keras.backend.clear_session()
            set_seed(base_seed + r)

            model = builder_fn(**cfg)
            metrics = train_eval_once(
                model, X_train_int, y_train, X_test_int, y_test
            )
            all_runs.append(metrics)

        # Agregamos media y desviación estándar por métrica.
        row = dict(cfg)
        for metric_name in [
            "accuracy",
            "auc",
            "best_val_accuracy",
            "best_val_auc",
            "train_time_s",
        ]:
            vals = [m[metric_name] for m in all_runs]
            row[f"{metric_name}_mean"] = float(np.mean(vals))
            row[f"{metric_name}_std"] = float(np.std(vals))
        rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results = df_results.sort_values(
        "auc_mean", ascending=False
    ).reset_index(drop=True)
    return df_results

In [9]:
# ===================================================
# Ejecutar búsqueda de hiperparámetros (E7)
# ===================================================
# Rejilla para Transformer con posición.
transformer_grid = [
    {
        "embed_dim": 32,
        "dense_dim": 64,
        "num_heads": 2,
        "dropout": 0.2,
        "lr": 1e-3,
        "use_positional": True,
    },
    {
        "embed_dim": 64,
        "dense_dim": 128,
        "num_heads": 4,
        "dropout": 0.2,
        "lr": 1e-3,
        "use_positional": True,
    },
    {
        "embed_dim": 64,
        "dense_dim": 256,
        "num_heads": 4,
        "dropout": 0.3,
        "lr": 5e-4,
        "use_positional": True,
    },
    {
        "embed_dim": 96,
        "dense_dim": 192,
        "num_heads": 6,
        "dropout": 0.2,
        "lr": 5e-4,
        "use_positional": True,
    },
]

# Rejilla para BiLSTM.
bilstm_grid = [
    {"embed_dim": 16, "lstm_units": 50, "dropout": 0.2, "lr": 1e-3},
    {"embed_dim": 32, "lstm_units": 64, "dropout": 0.2, "lr": 1e-3},
    {"embed_dim": 32, "lstm_units": 96, "dropout": 0.3, "lr": 5e-4},
    {"embed_dim": 64, "lstm_units": 128, "dropout": 0.3, "lr": 5e-4},
]

# Ejecutamos búsqueda para Transformer.
tr_results = run_grid(
    builder_fn=lambda **cfg: build_transformer_classifier(
        vocab_size=vocab_size, max_len=max_len, **cfg
    ),
    grid=transformer_grid,
    repeats=2,
)

# Ejecutamos búsqueda para BiLSTM.
bl_results = run_grid(
    builder_fn=lambda **cfg: build_bilstm_classifier(
        vocab_size=vocab_size, max_len=max_len, **cfg
    ),
    grid=bilstm_grid,
    repeats=2,
)

print("=== Top 3 Transformer (ordenado por auc_mean) ===")
print(tr_results.head(3).to_string(index=False))

print("\n=== Top 3 BiLSTM (ordenado por auc_mean) ===")
print(bl_results.head(3).to_string(index=False))

# Selección del mejor por AUC media en test.
best_tr = tr_results.iloc[0]
best_bl = bl_results.iloc[0]

print("\n=== Comparación final E7 ===")
print(
    f"Mejor AUC Transformer: {best_tr['auc_mean']:.4f} ± {best_tr['auc_std']:.4f}"
)
print(
    f"Mejor AUC BiLSTM     : {best_bl['auc_mean']:.4f} ± {best_bl['auc_std']:.4f}"
)
print(f"Tiempo medio Transformer (s): {best_tr['train_time_s_mean']:.1f}")
print(f"Tiempo medio BiLSTM (s)     : {best_bl['train_time_s_mean']:.1f}")

winner = (
    "Transformer con posición"
    if best_tr["auc_mean"] > best_bl["auc_mean"]
    else "BiLSTM"
)
print(f"Modelo ganador por AUC media: {winner}")

I0000 00:00:1775662949.648356   61772 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_22499__.56
I0000 00:00:1775662950.311405   65260 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_29', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1775662951.431209   65266 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 12 bytes spill stores, 24 bytes spill loads

I0000 00:00:1775662951.599272   65259 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 24 bytes spill stores, 28 bytes spill loads

I0000 00:00:1775662953.455552   65260 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1775662953.456659   65254 subprocess_compilation.cc:348]

=== Top 3 Transformer (ordenado por auc_mean) ===
 embed_dim  dense_dim  num_heads  dropout     lr  use_positional  accuracy_mean  accuracy_std  auc_mean  auc_std  best_val_accuracy_mean  best_val_accuracy_std  best_val_auc_mean  best_val_auc_std  train_time_s_mean  train_time_s_std
        96        192          6      0.2 0.0005            True       0.831447      0.005031   0.90736 0.001109                0.817610                    0.0           0.898844          0.000427          40.628311          9.072913
        32         64          2      0.2 0.0010            True       0.480503      0.000000   0.50000 0.000000                0.471698                    0.0           0.500000          0.000000          29.880362          7.391346
        64        128          4      0.2 0.0010            True       0.480503      0.000000   0.50000 0.000000                0.471698                    0.0           0.500000          0.000000          25.013919          0.168564

=== Top 3 BiL